# Exploring a single-band raster with `Dataset`

This tutorial opens a one-band MSWEP precipitation raster covering South America and uses it to tour the
read-only properties of a pyramids `Dataset`. You will learn how to inspect a raster's dimensions, band
metadata, geospatial referencing, driver details, and pixel values, then copy the dataset, add a band, read
windowed blocks, compute statistics, and attach an attribute table. Every attribute shown here is available
on any `Dataset`, whatever its file format.

## Load the raster

We import the `Dataset` class and point at the GeoTIFF — one day of MSWEP rainfall over South America.
The path is relative to this notebook.

In [ ]:
# NBVAL_IGNORE_OUTPUT
from pyramids.dataset import Dataset

%matplotlib inline

path = r"../../../examples/data/geotiff/south-america-mswep_1979010100.tif"

`read_file` opens the raster through GDAL; printing the returned `Dataset` gives a one-glance summary of its grid,
CRS, and bands.

In [ ]:
dataset = Dataset.read_file(path)
print(dataset)

Draw the band as a map so we can see the rainfall field we are about to inspect.

In [ ]:
dataset.plot()

The map shows daily rainfall across South America: wetter cells stand out over the tropical north and the Andes,
while much of the continent is dry on this particular day.

## Raster dimensions

The first thing to know about a raster is its grid geometry: the size of each pixel and how many rows and
columns it has. These properties describe the shape of the array without reading any pixel values.

In [ ]:
print(dataset.cell_size)

`rows` and `columns` report the grid height and width in pixels.

In [ ]:
print(f"Rows , Columns = {dataset.rows}, {dataset.columns}")

`shape` returns the array shape as `(bands, rows, columns)` — the same layout NumPy uses.

In [ ]:
print(dataset.shape)

`band_count` is the number of bands; this rainfall raster has just one.

In [ ]:
print(dataset.band_count)

## Band information

Each band carries its own metadata: a name, a data type, physical units, and optional scale/offset factors
used to decode stored numbers into real-world values.

In [ ]:
print(dataset.band_names)

`dtype` is the pixel data type (here a 32-bit float).

In [ ]:
print(dataset.dtype)

`band_units` lists the physical unit of each band, when the file records one.

In [ ]:
print(dataset.band_units)

`scale` is the multiplier applied when decoding stored values.

In [ ]:
print(dataset.scale)

`offset` is the constant added after scaling; together `scale` and `offset` map stored numbers to physical values.

In [ ]:
print(dataset.offset)

## Geospatial information

These properties tie the pixel grid to real-world locations: the affine geotransform, the bounding box, the
coordinate reference system, and the per-axis coordinate arrays.

In [ ]:
print(dataset.geotransform)
print(dataset.top_left_corner)

`bounds` returns the raster's extent as a `GeoDataFrame` — a one-row polygon of its footprint.

In [ ]:
print(dataset.bounds)

`bbox` returns the same extent as a plain `[xmin, ymin, xmax, ymax]` list.

In [ ]:
print(dataset.bbox)

`epsg` is the numeric CRS code; `crs` is its full WKT description.

In [ ]:
print(dataset.epsg)
print(dataset.crs)

`no_data_value` is the sentinel marking cells with no measurement — masked out in plots and statistics.

In [ ]:
print(dataset.no_data_value)

`lon` is the array of longitude coordinates for the grid columns.

In [ ]:
print(dataset.lon)

`lat` is the array of latitude coordinates for the grid rows.

In [ ]:
print(dataset.lat)

`x` gives the column coordinates in the dataset's native CRS units (identical to `lon` for a geographic raster).

In [ ]:
print(dataset.x)

`y` gives the row coordinates in native CRS units (identical to `lat` here).

In [ ]:
print(dataset.y)

## Driver and file metadata

Beyond the pixels, a `Dataset` knows how it is stored: the GDAL driver, the internal tiling/block size, the
source file name, and any key/value metadata tags.

In [ ]:
print(dataset.meta_data)

`block_size` is the internal tile size GDAL reads in one chunk — important for efficient windowed reads.

In [ ]:
print(dataset.block_size)

`file_name` is the path the dataset was opened from.

In [ ]:
print(dataset.file_name)

`driver_type` names the GDAL driver backing the dataset (GeoTIFF here).

In [ ]:
print(dataset.driver_type)

## Create a copy of the dataset

`copy` returns an independent in-memory clone, so you can modify it freely without touching the original
file on disk.

In [ ]:
dataset_copy = dataset.copy()
print(dataset_copy)

## Add a new band

`add_band` appends an extra band from a NumPy array. Here we stack a random array shaped to the grid as a
second band and give it a unit.

In [ ]:
import numpy as np

new_band = np.random.rand(dataset.rows, dataset.columns)
new_dataset = dataset.add_band(new_band, unit="mm")
print(new_dataset)

## Read band values

So far we only inspected metadata. `read_array` is what actually pulls pixel values into a NumPy array —
either the whole raster at once or a windowed block.

In [ ]:
arr = dataset.read_array()
print(arr.shape)

Display the array itself to see the raw rainfall values, including the no-data cells.

In [ ]:
arr

Most cells hold small floating-point rainfall values; the large negative entries are the no-data sentinel seen above.

`read_array` with a `window=(x, y, width, height)` reads only a sub-block — here the top-left 100x100 pixels —
instead of loading the whole raster into memory.

In [ ]:
block = dataset.read_array(band=0, window=(0, 0, 100, 100))
print(block.shape)
print(block)

`get_block_arrangement` lists how the raster would be tiled into blocks of the given size, handy for chunked,
memory-safe processing of large rasters.

In [ ]:
dataset.get_block_arrangement(x_block_size=100, y_block_size=100)

## Band statistics

`stats` computes summary statistics (min, max, mean, standard deviation) for a band, skipping no-data cells.

In [ ]:
stats = dataset.stats(band=0)
stats

The statistics summarise the whole band in one row — a quick way to read the value range and spot outliers before analysis.

## Attribute table

A raster band can carry a raster attribute table (RAT) — a small lookup table that gives meaning to pixel
values, much like a map legend. Below we read the current table, then set our own.

### Read the band attribute table

In [ ]:
df = dataset.get_attribute_table(band=0)
print(df)

The raster ships without a predefined attribute table, so the returned frame is empty — we populate it next.

### Set the band attribute table

`set_attribute_table` attaches a pandas DataFrame to the band, labelling ranges of pixel values with a
category and description — turning bare numbers into an interpretable legend.

In [ ]:
import pandas as pd

attribute_table = {
    'Precipitation Range (mm)': ['0-50', '51-100', '101-200', '201-500', '>500'],
    'Category': ['Low', 'Moderate', 'High', 'Very High', 'Extreme'],
    'Description': [
        'Very low precipitation',
        'Moderate precipitation',
        'High precipitation',
        'Very high precipitation',
        'Extreme precipitation',
    ],
}
df = pd.DataFrame(attribute_table)
dataset.set_attribute_table(df, band=0)